# Trabajo Práctico Integrador — Entrega 2
## Ciencia de Datos 2026 | UTN FRC | 5K4 | Grupo 15

---

### Descripción de la entrega

Este notebook implementa el pipeline **ETL (Extract, Transform, Load)** completo sobre el dataset PAMAP2, correspondiente a la **Entrega 2** del TPI. Incluye:

- Carga y unificación de los 9 archivos fuente (.dat)
- Downsampling 100 Hz → 10 Hz
- Eliminación de columnas inválidas (orientación IMU)
- Exclusión de períodos transitorios (activityID = 0)
- Tratamiento de valores faltantes en heart\_rate
- Verificación y guardado del dataset limpio

---

### Integrantes — Grupo 15

| Rol | Nombre | Legajo |
|-----|--------|--------|
| Product Owner | Franco Recalde | 94661 |
| Scrum Master | Emilio Sadir | — |
| Team | (6 integrantes restantes) | — |

---

### Dataset

**PAMAP2 Physical Activity Monitoring Dataset**  
Fuente: [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/231/pamap2+physical+activity+monitoring)  
Licencia: CC BY 4.0  
Autores: A. Reiss y D. Stricker (DFKI, Alemania)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid')

print('Librerias cargadas correctamente.')

## Configuración del entorno

La notebook detecta automáticamente si se ejecuta en **Google Colab** o localmente.
En Colab se monta Google Drive y se asume que la carpeta del proyecto está en `MyDrive/TP_CD/`.
Ajustar `BASE_PATH` si la ubicación en Drive es diferente.

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/TP_CD')
else:
    BASE_PATH = Path('..').resolve()

DATA_PROTOCOL = BASE_PATH / 'PAMAP2_Dataset' / 'Protocol'
DATA_OPTIONAL = BASE_PATH / 'PAMAP2_Dataset' / 'Optional'
OUTPUT_PATH   = BASE_PATH / 'data' / 'processed'
FIGURES_PATH  = BASE_PATH / 'reports' / 'entrega_2' / 'figures'

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
FIGURES_PATH.mkdir(parents=True, exist_ok=True)

env_name = 'Google Colab' if IN_COLAB else 'Local'
print(f'Entorno:   {env_name}')
print(f'Datos en:  {DATA_PROTOCOL}')
print(f'Output en: {OUTPUT_PATH}')

## Definición de columnas

El dataset PAMAP2 no incluye header en sus archivos `.dat`. Las **54 columnas** se asignan
manualmente según la documentación oficial (readme.pdf, UCI). Se identifican las
**12 columnas de orientación** (4 por IMU × 3 IMUs: hand, chest, ankle) que serán
eliminadas en el Paso 3 por ser inválidas según la documentación oficial.

In [ ]:
RAW_COLUMN_NAMES = [
    'timestamp', 'activity_id', 'heart_rate',
    # IMU hand — 17 columnas
    'hand_temp',
    'hand_acc1_x', 'hand_acc1_y', 'hand_acc1_z',
    'hand_acc2_x', 'hand_acc2_y', 'hand_acc2_z',
    'hand_gyro_x', 'hand_gyro_y', 'hand_gyro_z',
    'hand_mag_x',  'hand_mag_y',  'hand_mag_z',
    'hand_orient_1', 'hand_orient_2', 'hand_orient_3', 'hand_orient_4',
    # IMU chest — 17 columnas
    'chest_temp',
    'chest_acc1_x', 'chest_acc1_y', 'chest_acc1_z',
    'chest_acc2_x', 'chest_acc2_y', 'chest_acc2_z',
    'chest_gyro_x', 'chest_gyro_y', 'chest_gyro_z',
    'chest_mag_x',  'chest_mag_y',  'chest_mag_z',
    'chest_orient_1', 'chest_orient_2', 'chest_orient_3', 'chest_orient_4',
    # IMU ankle — 17 columnas
    'ankle_temp',
    'ankle_acc1_x', 'ankle_acc1_y', 'ankle_acc1_z',
    'ankle_acc2_x', 'ankle_acc2_y', 'ankle_acc2_z',
    'ankle_gyro_x', 'ankle_gyro_y', 'ankle_gyro_z',
    'ankle_mag_x',  'ankle_mag_y',  'ankle_mag_z',
    'ankle_orient_1', 'ankle_orient_2', 'ankle_orient_3', 'ankle_orient_4',
]

ORIENT_COLS = [col for col in RAW_COLUMN_NAMES if 'orient' in col]

print(f'Total columnas raw: {len(RAW_COLUMN_NAMES)}')
print(f'Columnas de orientacion a eliminar ({len(ORIENT_COLS)}):')
for col in ORIENT_COLS:
    print(f'  - {col}')

---
## Paso 1: Carga y unificación de los archivos fuente

Se cargan los 9 archivos `.dat` del protocolo estándar (subject101 a subject109). Cada archivo:
- No tiene header
- Usa espacios como separador (uno o varios)
- Contiene exactamente 54 columnas

Se agrega la columna `subject_id` al momento de la carga. Esta columna es **esencial**
para la Entrega 3, donde se aplicará cross-validation leave-one-subject-out (LOSO-CV),
estándar en literatura HAR para evitar data leakage entre sujetos.

In [ ]:
def load_subject(filepath, subject_id):
    df = pd.read_csv(
        filepath,
        sep=r'\s+',
        header=None,
        names=RAW_COLUMN_NAMES,
        engine='python'
    )
    df['subject_id'] = subject_id
    return df

dfs = []
for subject_id in range(101, 110):
    filepath = DATA_PROTOCOL / f'subject{subject_id}.dat'
    df_subj = load_subject(filepath, subject_id)
    dfs.append(df_subj)
    n_act = df_subj['activity_id'].nunique()
    print(f'  subject{subject_id}: {len(df_subj):>9,} filas | {n_act} actividades')

df_raw = pd.concat(dfs, ignore_index=True)
print()
print(f'DataFrame unificado: {df_raw.shape[0]:,} filas x {df_raw.shape[1]} columnas')
print(f'Memoria:             {df_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB')

---
## Paso 2: Downsampling 100 Hz → 10 Hz

Las IMU muestrean a **100 Hz**, generando ~3.8 millones de filas en bruto. Se aplica
downsampling tomando **1 fila cada 10** por sujeto y por actividad (`iloc[::10]`).

**Justificación:**
- 10 Hz captura los patrones de movimiento relevantes para HAR (actividades cotidianas
  no superan los 5-6 Hz en contenido de señal útil)
- Reduce el volumen ~10×: de ~3.8M a ~194K filas
- Estándar en múltiples trabajos del estado del arte sobre PAMAP2
- Se aplica por sujeto **y** por actividad para no mezclar señales de sesiones distintas

In [ ]:
def downsample(df, step=10):
    return (
        df.groupby(['subject_id', 'activity_id'], group_keys=False)
          .apply(lambda g: g.iloc[::step])
    )

df_ds = downsample(df_raw, step=10).reset_index(drop=True)

factor = len(df_raw) / len(df_ds)
print(f'Antes del downsampling:   {len(df_raw):>9,} filas')
print(f'Despues del downsampling: {len(df_ds):>9,} filas')
print(f'Factor de reduccion:      {factor:.1f}x')
print(f'Memoria:                  {df_ds.memory_usage(deep=True).sum() / 1e6:.1f} MB')
print()
print('Distribucion por sujeto tras downsampling:')
print(df_ds.groupby('subject_id').size().rename('filas').to_frame().to_string())

---
## Paso 3: Eliminación de columnas de orientación

Según la documentación oficial del dataset PAMAP2 (`readme.pdf`, UCI), las columnas de
orientación de los tres sensores IMU son **inválidas** en esta recolección. La calibración
de los quaterniones no fue completada correctamente durante la toma de datos.

Se eliminan las **12 columnas** de orientación (4 por IMU × 3 IMUs: hand, chest, ankle),
reduciendo el dataset de 55 → 43 columnas.

In [ ]:
print('Columnas eliminadas:')
for col in ORIENT_COLS:
    print(f'  - {col}')

df_no_orient = df_ds.drop(columns=ORIENT_COLS)

print()
print(f'Columnas antes:   {df_ds.shape[1]}')
print(f'Columnas despues: {df_no_orient.shape[1]}  (menos {df_ds.shape[1] - df_no_orient.shape[1]})')
print(f'Shape resultante: {df_no_orient.shape}')

---
## Paso 4: Exclusión de activityID == 0

El valor `activity_id = 0` está etiquetado como *"other"* en el dataset y corresponde a
**períodos transitorios** entre actividades. Estos registros:
- No representan ninguna actividad definida del protocolo
- Introducen ambigüedad en los patrones de señal
- Son ruido para el modelo de clasificación

Se excluyen completamente del dataset de trabajo.

In [ ]:
filas_antes = len(df_no_orient)
df_excl = df_no_orient[df_no_orient['activity_id'] != 0].copy()
filas_despues = len(df_excl)
eliminadas = filas_antes - filas_despues

print(f'Filas antes:                {filas_antes:>9,}')
print(f'Filas con activity_id == 0: {eliminadas:>9,}  ({eliminadas/filas_antes*100:.1f}%)')
print(f'Filas despues:              {filas_despues:>9,}')
print()
acts = sorted(df_excl['activity_id'].unique())
n_acts = df_excl['activity_id'].nunique()
print(f'Actividades presentes ({n_acts}): {acts}')

---
## Paso 5: Tratamiento de NaNs en heart_rate

El sensor de frecuencia cardíaca (BM-CS5X) muestrea a **~9 Hz**, mientras que las IMUs
muestrean a 100 Hz. Esto genera **NaN frecuentes y estructurales** en `heart_rate`:
no son errores de medición sino una consecuencia directa de la diferencia de frecuencias.

**Estrategia: interpolación lineal por sujeto + fill en bordes**

- No se usa media/mediana global: incorrecto para una serie temporal fisiológica continua
- Se interpola linealmente dentro de cada `subject_id` para respetar la continuidad temporal
- Los NaN en bordes (inicio/fin de sesión) se resuelven con forward-fill y backward-fill
- El `groupby` por sujeto garantiza que no se mezclen señales entre participantes

In [ ]:
nan_antes = df_excl['heart_rate'].isna().sum()
pct_antes = nan_antes / len(df_excl) * 100
print(f'NaNs en heart_rate ANTES:   {nan_antes:,}  ({pct_antes:.1f}%)')

df_clean = df_excl.copy()
df_clean['heart_rate'] = (
    df_clean.groupby('subject_id')['heart_rate']
            .transform(lambda x: x.interpolate(method='linear').ffill().bfill())
)

nan_despues = df_clean['heart_rate'].isna().sum()
print(f'NaNs en heart_rate DESPUES: {nan_despues:,}')
print(f'NaNs resueltos:             {nan_antes - nan_despues:,}')

---
## Verificación final del dataset limpio

Se verifica el estado del DataFrame resultante: shape, tipos de datos y porcentaje
de nulos por columna. El dataset debe tener **0 nulos** antes de ser guardado.

In [ ]:
print('=' * 60)
print('VERIFICACION FINAL DEL DATASET LIMPIO')
print('=' * 60)

print(f'Shape: {df_clean.shape[0]:,} filas x {df_clean.shape[1]} columnas')

n_acts = df_clean['activity_id'].nunique()
print(f'Actividades presentes ({n_acts}):')
print(' ', sorted(df_clean['activity_id'].unique()))

n_subjs = df_clean['subject_id'].nunique()
print(f'Sujetos ({n_subjs}):')
print(' ', sorted(df_clean['subject_id'].unique()))

print()
print('DTYPES:')
print(df_clean.dtypes.to_string())

print()
pct_nulos = df_clean.isnull().mean() * 100
nulos_nonzero = pct_nulos[pct_nulos > 0]
if len(nulos_nonzero) > 0:
    print('% NULOS POR COLUMNA (solo columnas con nulos):')
    print(nulos_nonzero.sort_values(ascending=False).to_string())
else:
    print('Sin valores nulos en el dataset final.')

print()
print('ESTADISTICOS DESCRIPTIVOS:')
display(df_clean.describe().round(4))

In [ ]:
output_file = OUTPUT_PATH / 'pamap2_clean.csv'
df_clean.to_csv(output_file, index=False)
print(f'Dataset guardado en: {output_file}')
print(f'Shape final:         {df_clean.shape}')
print()
print(f'Columnas ({df_clean.shape[1]}):')
for col in df_clean.columns:
    print(f'  {col}')